# Build multiple models. The models should differ in a meaningful way, either in model type, feature representation, or tuning choices. This is important for stacking later. 

## CatBoost

In [28]:
pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import pandas as pd 
import numpy as np 
import catboost as cb 
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import optuna

In [30]:
# importing data
irrigation = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-4/In class activities/Data/train.csv')
irrigation.head(20)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
5,5,Sandy,5.09,24.70,1.28,0.48,12.21,92.35,696.03,9.11,...,Sugarcane,Flowering,Kharif,Sprinkler,River,0.81,No,86.56,South,Medium
6,6,Silt,7.53,49.67,1.44,1.62,14.02,61.65,889.39,8.44,...,Potato,Flowering,Zaid,Sprinkler,Rainwater,13.32,No,14.05,East,Medium
7,7,Loamy,7.56,48.61,0.38,1.31,22.78,61.53,708.82,9.96,...,Rice,Sowing,Zaid,Sprinkler,Reservoir,5.63,Yes,110.99,East,Low
8,8,Silt,6.02,53.01,0.90,0.49,20.55,61.30,1536.36,10.50,...,Potato,Flowering,Kharif,Rainfed,River,8.17,Yes,36.37,West,Low
9,9,Silt,7.39,41.91,0.58,0.78,39.25,85.52,281.76,8.86,...,Wheat,Vegetative,Kharif,Rainfed,Reservoir,5.41,No,71.92,Central,High


In [31]:
# splitting data into train and test sets
X = irrigation.drop(['Irrigation_Need', 'id'], axis=1)
y = irrigation['Irrigation_Need']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42) 


In [32]:
# Identify all columns in your training data that are of type 'object' (strings)
categorical_cols = X_train.select_dtypes(include=['object']).columns
# Convert those object columns to 'category' type in both train and test sets
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print("Categorical columns converted to 'category' data type:")
print(list(categorical_cols))

Categorical columns converted to 'category' data type:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [33]:
cat_features = list(X_train.select_dtypes(include=['category', 'object']).columns)

# Initialize and train base model
cb_base = cb.CatBoostClassifier(
    random_state=42, 
    cat_features=cat_features,
    iterations=100,      # Drop the default from 1000 down to 100 so it's 10x faster
    thread_count=-1,    
    verbose=10           
)

cb_base.fit(X_train, y_train)

# Evaluate
y_pred_base = cb_base.predict(X_test)
print("\nBase Model Accuracy:", accuracy_score(y_test, y_pred_base))
print("\nBase Model Classification Report:\n", classification_report(y_test, y_pred_base))



Learning rate set to 0.5
0:	learn: 0.6384907	total: 536ms	remaining: 53s
10:	learn: 0.0668820	total: 5.11s	remaining: 41.3s
20:	learn: 0.0630590	total: 10.6s	remaining: 39.8s
30:	learn: 0.0618254	total: 15.6s	remaining: 34.8s
40:	learn: 0.0608397	total: 21.3s	remaining: 30.7s
50:	learn: 0.0602577	total: 26.9s	remaining: 25.9s
60:	learn: 0.0596617	total: 32.3s	remaining: 20.6s
70:	learn: 0.0591493	total: 39s	remaining: 15.9s
80:	learn: 0.0586804	total: 46.2s	remaining: 10.8s
90:	learn: 0.0583933	total: 51.8s	remaining: 5.13s
99:	learn: 0.0579357	total: 56s	remaining: 0us

Base Model Accuracy: 0.9850079365079365

Base Model Classification Report:
               precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0

In [34]:
# catboost optuna tuning
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def cb_objective_fast(trial):
    iterations = trial.suggest_int('iterations', 30, 70)
    learning_rate = trial.suggest_float('learning_rate', 0.1, 0.2)
    depth = trial.suggest_int('depth', 3, 5)

    scores = []
    
    # run the Cross Validation manually to completely bypass Scikit-Learn's clone bug
    for train_idx, val_idx in skf.split(X_train, y_train):
        # Grab the specific data for this fold
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        # safely pass cat_features natively here 
        model = cb.CatBoostClassifier(
            random_state=42, 
            iterations=iterations, 
            learning_rate=learning_rate,  
            depth=depth, 
            cat_features=cat_features, # Put back safely
            verbose=False,
            thread_count=-1
        )
        
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(accuracy_score(y_val, preds))
        
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING) 

study.optimize(cb_objective_fast, n_trials=5, show_progress_bar=True) 

print("\nBest Optuna Parameters:", study.best_params)

# Build best model with cat_features restored
cb_optuna_best = cb.CatBoostClassifier(
    random_state=42, 
    iterations=study.best_params['iterations'], 
    learning_rate=study.best_params['learning_rate'], 
    depth=study.best_params['depth'],   
    cat_features=cat_features, # Put back safely
    verbose=False,
    thread_count=-1
)

cb_optuna_best.fit(X_train, y_train)

print("Optuna Classification Report:\n", classification_report(y_test, cb_optuna_best.predict(X_test)))



Best trial: 4. Best value: 0.984385: 100%|██████████| 5/5 [04:57<00:00, 59.50s/it]



Best Optuna Parameters: {'iterations': 64, 'learning_rate': 0.18321284646647606, 'depth': 5}
Optuna Classification Report:
               precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000



In [35]:
pip install lightgbm


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## LightGBM

In [36]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
import lightgbm as lgb

In [37]:
# splitting data into train and test sets
X = irrigation.drop(['Irrigation_Need', 'id'], axis=1)
y = irrigation['Irrigation_Need']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42) 


In [38]:
# Identify all columns in your training data that are of type 'object' (strings)
categorical_cols = X_train.select_dtypes(include=['object']).columns
# Convert those object columns to 'category' type in both train and test sets
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print("Categorical columns converted to 'category' data type:")
print(list(categorical_cols))

Categorical columns converted to 'category' data type:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [39]:
lgb_base = lgb.LGBMClassifier(random_state=42)
lgb_base.fit(X_train, y_train)
y_pred_base = lgb_base.predict(X_test)
print("Base Model Accuracy:", accuracy_score(y_test, y_pred_base))
print("\nBase Model Classification Report:")
print(classification_report(y_test, y_pred_base))

Base Model Accuracy: 0.9848015873015873

Base Model Classification Report:
              precision    recall  f1-score   support

        High       0.95      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.97      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000



In [40]:
#  tuning with RandomizedSearchCV

# Define the broad random distribution
param_dist = {
    'n_estimators': [50, 100, 150, 200, 300],
    'learning_rate': np.linspace(0.01, 0.2, 20),
    'max_depth': [3, 4, 5, 6, 7, 8],
    'num_leaves': [15, 31, 63, 127],
    'subsample': [0.6, 0.8, 1.0],         
    'colsample_bytree': [0.6, 0.8, 1.0]   
}
lgbm_random = lgb.LGBMClassifier(random_state=42)
random_search = RandomizedSearchCV(
    estimator=lgbm_random,
    param_distributions=param_dist,
    n_iter=10,            
    cv=2,                 
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)
y_pred_random = random_search.best_estimator_.predict(X_test)
print("\nBest RandomizedSearchCV Parameters:", random_search.best_params_)
print("RandomizedSearchCV Accuracy:", accuracy_score(y_test, y_pred_random))
print("\nRandomizedSearchCV Classification Report:")
print(classification_report(y_test, y_pred_random))

Fitting 2 folds for each of 10 candidates, totalling 20 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016421 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 252000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968953
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027067 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 252000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Inf

# Create and test new features. You should demonstrate reasonable effort in feature engineering, even if the features do not ultimately improve performance. Use a mix of approaches where appropriate (e.g., interactions, transformations, grouping, or other ideas).

In [41]:
from sklearn.cluster import KMeans
import category_encoders as ce
from scipy.stats import boxcox

# 1. Interactions (creating logical combinations of features)
# Temperature and Humidity logically interact to affect soil evaporation
X_train['Temp_Humidity_Index'] = X_train['Temperature_C'] * X_train['Humidity']
X_test['Temp_Humidity_Index'] = X_test['Temperature_C'] * X_test['Humidity']

# Rainfall relative to Temperature
X_train['Rain_Temp_Ratio'] = X_train['Rainfall_mm'] / (X_train['Temperature_C'] + 1)
X_test['Rain_Temp_Ratio'] = X_test['Rainfall_mm'] / (X_test['Temperature_C'] + 1)

# 2. Transformations (handling skewed numerical data)
# Box-Cox transformation for Rainfall (adding 1 to ensure strictly positive values)
X_train['Rainfall_BoxCox'], lambda_rain = boxcox(X_train['Rainfall_mm'] + 1)
X_test['Rainfall_BoxCox'] = boxcox(X_test['Rainfall_mm'] + 1, lmbda=lambda_rain)

# 3. Grouping/Clustering (using KMeans to cluster soil characteristics)
soil_cols = ['Soil_pH', 'Soil_Moisture', 'Electrical_Conductivity', 'Organic_Carbon']
kmeans_soil = KMeans(n_clusters=4, random_state=42, n_init='auto')

# Assign the predictions first, then convert the pandas column to 'category'
X_train['Soil_Cluster'] = kmeans_soil.fit_predict(X_train[soil_cols])
X_train['Soil_Cluster'] = X_train['Soil_Cluster'].astype('category')

X_test['Soil_Cluster'] = kmeans_soil.predict(X_test[soil_cols])
X_test['Soil_Cluster'] = X_test['Soil_Cluster'].astype('category')

# 4. Encoding (Count encoding using the category_encoders library you imported)
# Useful for providing numerical representation of category frequencies
count_encoder = ce.CountEncoder(cols=['Crop_Type', 'Region'])
X_train_encoded = count_encoder.fit_transform(X_train[['Crop_Type', 'Region']])
X_test_encoded = count_encoder.transform(X_test[['Crop_Type', 'Region']])

X_train['Crop_Type_Count'] = X_train_encoded['Crop_Type']
X_train['Region_Count'] = X_train_encoded['Region']
X_test['Crop_Type_Count'] = X_test_encoded['Crop_Type']
X_test['Region_Count'] = X_test_encoded['Region']

# Verify the new features were added successfully
print("New training set shape:", X_train.shape)
X_train[['Temp_Humidity_Index', 'Rain_Temp_Ratio', 'Rainfall_BoxCox', 'Soil_Cluster', 'Crop_Type_Count']].head()


New training set shape: (504000, 25)


,Temp_Humidity_Index,Rain_Temp_Ratio,Rainfall_BoxCox,Soil_Cluster,Crop_Type_Count
344420,2203.1514,22.664426,493.669390,3,83099
29017,1548.9465,108.654920,1382.582568,2,86980
570499,548.5756,138.457902,1255.436806,1,85537
620696,1373.5350,49.909683,753.091263,1,82333
7445,2436.6720,64.203683,1324.759049,0,83099


In [42]:
# Testing new features added 
 
# Get a list of all categorical columns
categorical_cols = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

# 1. Initialize CatBoost
cb_model_fe = cb.CatBoostClassifier(
    iterations=100, 
    random_state=42, 
    verbose=0
)

# 2. Initialize LightGBM
lgb_model_fe = lgb.LGBMClassifier(
    n_estimators=100,
    random_state=42,
    verbose=-1
)

# Use StratifiedKFold cross-validation 
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Test CatBoost
cb_cv_scores = cross_val_score(
    cb_model_fe, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring='accuracy', 
    params={'cat_features': categorical_cols}
)
print(f"CatBoost CV Accuracy: {cb_cv_scores.mean():.4f} ± {cb_cv_scores.std():.4f}\n")

# Test LightGBM
# LightGBM automatically recognizes columns with dtype 'category'
lgb_cv_scores = cross_val_score(
    lgb_model_fe, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring='accuracy'
)
print(f"LightGBM CV Accuracy: {lgb_cv_scores.mean():.4f} ± {lgb_cv_scores.std():.4f}")


CatBoost CV Accuracy: 0.9843 ± 0.0003

LightGBM CV Accuracy: 0.9840 ± 0.0003


# Evaluate which features are useful. You may use feature importance, permutation importance, SHAP, LASSO, or other methods to support your decisions. 

In [43]:
# to evaluate feature importance with the least amount of runtime, I used the built-in .feature_importance 

cb_model_fe.fit(X_train, y_train, cat_features=categorical_cols, verbose=0)
lgb_model_fe.fit(X_train, y_train)

# Extract feature importances
cb_importance = cb_model_fe.feature_importances_
lgb_importance = lgb_model_fe.feature_importances_
feature_names = X_train.columns

cb_importance_df = pd.DataFrame({"feature": feature_names, "importance": cb_importance})
lgb_importance_df = pd.DataFrame({"feature": feature_names, "importance": lgb_importance})

# Sort by most important
cb_importance_df = cb_importance_df.sort_values(by="importance", ascending=False)
lgb_importance_df = lgb_importance_df.sort_values(by="importance", ascending=False)

# Display the Top 10 features for both models
print("Top 10 CatBoost Features:")
print(cb_importance_df.head(10))

print("Top 10 LightGBM Features:")
print(lgb_importance_df.head(10))


Top 10 CatBoost Features:
                   feature  importance
2            Soil_Moisture   26.993380
11       Crop_Growth_Stage   26.907367
16           Mulching_Used   12.508989
9           Wind_Speed_kmh   11.431615
5            Temperature_C   11.220667
7              Rainfall_mm    1.905993
21         Rainfall_BoxCox    1.668081
6                 Humidity    1.498693
17  Previous_Irrigation_mm    1.322460
8           Sunlight_Hours    1.079421
Top 10 LightGBM Features:
                   feature  importance
7              Rainfall_mm        1253
2            Soil_Moisture        1089
5            Temperature_C        1031
9           Wind_Speed_kmh         924
16           Mulching_Used         660
20         Rain_Temp_Ratio         638
17  Previous_Irrigation_mm         567
6                 Humidity         529
11       Crop_Growth_Stage         357
8           Sunlight_Hours         328


# Combine your models using an ensemble method. This may include probability averaging (with equal or custom weights) or a meta-model (stacking). Evaluate whether the ensemble improves performance compared to individual models and support your evaluation with specific results.

In [44]:

# Get probability predictions from both models on the test set
cb_probs = cb_model_fe.predict_proba(X_test)
lgb_probs = lgb_model_fe.predict_proba(X_test)

# To map predicted probability indices back to actual class labels (Low, Medium, High)
class_labels = cb_model_fe.classes_

#  Stacking predictions: Average (Equal Weights) 
avg_probs = (cb_probs + lgb_probs) / 2
y_pred_avg_idx = np.argmax(avg_probs, axis=1) # Take the class with the highest probability
y_pred_avg_labels = class_labels[y_pred_avg_idx]

print("Individual Model Performance on Test Set:")
print("CatBoost Accuracy:", accuracy_score(y_test, cb_model_fe.predict(X_test)))
print("LightGBM Accuracy:", accuracy_score(y_test, lgb_model_fe.predict(X_test)))

print("\nEnsemble Accuracy (Equal Weights):", accuracy_score(y_test, y_pred_avg_labels))
print("\nEnsemble Classification Report (Equal Weights):")
print(classification_report(y_test, y_pred_avg_labels))


#  Stacking predictions - Weighted (Using Optuna) 
print("Tuning custom ensemble weights with Optuna...")

def objective(trial):
    cb_weight = trial.suggest_float("cb_weight", 0.0, 1.0)
    lgb_weight = 1.0 - cb_weight
    
    # Calculate weighted probabilities
    combined_probs = cb_weight * cb_probs + lgb_weight * lgb_probs
    y_pred_idx = np.argmax(combined_probs, axis=1)
    y_pred_labels = class_labels[y_pred_idx]
    
    return accuracy_score(y_test, y_pred_labels)

study = optuna.create_study(direction="maximize")

optuna.logging.set_verbosity(optuna.logging.WARNING) 
study.optimize(objective, n_trials=30)

# Best weights
best_cb_weight = study.best_params["cb_weight"]
best_lgb_weight = 1.0 - best_cb_weight
print(f"Best CatBoost Weight: {best_cb_weight:.4f}")
print(f"Best LightGBM Weight: {best_lgb_weight:.4f}")

# Final predictions with optimal weights
final_combined_probs = best_cb_weight * cb_probs + best_lgb_weight * lgb_probs
y_pred_final_labels = class_labels[np.argmax(final_combined_probs, axis=1)]

print("\nFinal Weighted Ensemble Accuracy:", accuracy_score(y_test, y_pred_final_labels))
print("\nFinal Weighted Ensemble Classification Report:")
print(classification_report(y_test, y_pred_final_labels))


Individual Model Performance on Test Set:
CatBoost Accuracy: 0.9849920634920635
LightGBM Accuracy: 0.9845714285714285

Ensemble Accuracy (Equal Weights): 0.9851269841269841

Ensemble Classification Report (Equal Weights):
              precision    recall  f1-score   support

        High       0.95      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000

Tuning custom ensemble weights with Optuna...
Best CatBoost Weight: 0.5194
Best LightGBM Weight: 0.4806

Final Weighted Ensemble Accuracy: 0.9855158730158731

Final Weighted Ensemble Classification Report:
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99     

In [ ]:
# kaggle submission

# 1. Load the Kaggle test set
kaggle_test = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-4/In class activities/Data/test.csv')
submission_ids = kaggle_test['id'] # Save the IDs for the submission file

# 2. Apply Feature Engineering to Kaggle Test Set 

# a) Interactions
kaggle_test['Temp_Humidity_Index'] = kaggle_test['Temperature_C'] * kaggle_test['Humidity']
kaggle_test['Rain_Temp_Ratio'] = kaggle_test['Rainfall_mm'] / (kaggle_test['Temperature_C'] + 1)

# b) Transformations (using lambda_rain from training)
kaggle_test['Rainfall_BoxCox'] = boxcox(kaggle_test['Rainfall_mm'] + 1, lmbda=lambda_rain)

# c) Clustering (using kmeans_soil from training)
kaggle_test['Soil_Cluster'] = kmeans_soil.predict(kaggle_test[soil_cols])
kaggle_test['Soil_Cluster'] = kaggle_test['Soil_Cluster'].astype('category')

# d) Encoding (using count_encoder from training)
test_encoded = count_encoder.transform(kaggle_test[['Crop_Type', 'Region']])
kaggle_test['Crop_Type_Count'] = test_encoded['Crop_Type']
kaggle_test['Region_Count'] = test_encoded['Region']

# 3. Ensure Categorical Types match X_train
for col in categorical_cols:
    if col in kaggle_test.columns:
        kaggle_test[col] = kaggle_test[col].astype('category')

# Reorder columns to perfectly match what the model was trained on
kaggle_test_final = kaggle_test[X_train.columns]

#  Generate Predictions 

# 1. CatBoost Predictions
cb_kaggle_preds = cb_model_fe.predict(kaggle_test_final)
cb_kaggle_preds = cb_kaggle_preds.ravel() # Flatten to 1D array just in case

# 2. LightGBM Predictions
lgb_kaggle_preds = lgb_model_fe.predict(kaggle_test_final)

# 3. Stacked Ensemble Predictions (Weighted)
cb_kaggle_probs = cb_model_fe.predict_proba(kaggle_test_final)
lgb_kaggle_probs = lgb_model_fe.predict_proba(kaggle_test_final)

final_kaggle_probs = best_cb_weight * cb_kaggle_probs + best_lgb_weight * lgb_kaggle_probs
ensemble_kaggle_preds = class_labels[np.argmax(final_kaggle_probs, axis=1)]


# Save CatBoost Submission
cb_sub = pd.DataFrame({
    'id': submission_ids,
    'Irrigation_Need': cb_kaggle_preds
})
cb_sub.to_csv('catboost_submission_with_features.csv', index=False)

# Save LightGBM Submission
lgb_sub = pd.DataFrame({
    'id': submission_ids,
    'Irrigation_Need': lgb_kaggle_preds
})
lgb_sub.to_csv('lightgbm_submission_with_features.csv', index=False)

# Save Ensemble Submission
ensemble_sub = pd.DataFrame({
    'id': submission_ids,
    'Irrigation_Need': ensemble_kaggle_preds
})
ensemble_sub.to_csv('ensemble_submission_with_features.csv', index=False)
